# JCMIL: P3 + Joint
Chọn runtime GPU. Code tải từ GitHub theo commit cố định; dữ liệu và kết quả ở `/content`. Full-data, seed 0, 10 epoch, batch32, FP32. Chưa xác minh tốc độ/VRAM trên T4. Dữ liệu chính thức có ảnh trùng giữa split.

Dùng bản này cho lượt mới; ảnh tải dở trong `/content/crisismmd_media` được dùng tiếp. Không đổi code giữa lượt train.

In [ ]:
import os,sys,subprocess
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='0'
REVISION='47834609667ee78fa2cf0c0dee38ef60136b1fda'
ROOT=Path('/content/jcmil-code-'+REVISION[:12])
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/Thanh-000/jcmil-colab.git',str(ROOT)],check=True)
    subprocess.run(['git','checkout','--detach',REVISION],cwd=ROOT,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()==REVISION
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=ROOT,text=True).strip(), 'Code đã thay đổi'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')],check=True)
sys.path.insert(0,str(ROOT))
import torch
assert torch.cuda.is_available(), 'Chọn runtime GPU'
print(subprocess.check_output(['nvidia-smi'],text=True))
import colab as job
job.setup()

In [ ]:
job.download_data()

In [ ]:
job.prepare()

## Huấn luyện
Chạy lần lượt. Chạy lại ô bị ngắt để resume epoch hoàn tất. Payload checkpoint cũ được dọn sau khi xác minh checkpoint kế tiếp. Tải kết quả trước khi xóa runtime.

In [ ]:
job.train_job(1)

In [ ]:
job.train_job(2)

In [ ]:
job.train_job(3)

In [ ]:
job.train_job(4)

In [ ]:
job.train_job(1, joint=True)

In [ ]:
job.evaluate('dev')

## Test sau khi khóa recipe
Đổi `RUN_TEST` sau khi chốt cấu hình; không tối ưu theo test.

In [ ]:
RUN_TEST=False
if RUN_TEST:
    import json
    (job.RUN/'recipe-frozen-before-test.json').write_text(json.dumps({'commit':REVISION,'epochs':10,'batch_size':32,'precision':'fp32','seed':0}))
    job.evaluate('test')

In [ ]:
from google.colab import files
files.download(job.export_evidence())

In [ ]:
CHECKPOINT='s4'  # s1, s2, s3, s4, joint
files.download(job.export_checkpoint(CHECKPOINT))